# Exercise 3: Add Decorators for Logging and Timing

**Objectives**
- Learn the **Decorator Pattern** to add reusable behaviour (logging, timing).
- Understand how decorators support the **Open/Closed Principle**: classes stay *closed for modification* but *open for extension*.

**Concept:** a decorator is a function that takes a function, wraps it with extra behaviour, and returns the wrapper. The original method body is never edited.

In [1]:
import sys
import time
import logging
import functools
import pandas as pd

# Configure logging once so the decorators print to the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,   # same stream as print() so output stays in order
    force=True,
)
logger = logging.getLogger("pipeline")

## Step 1 — Create the two decorators

- `@log_action` → logs when a method **starts** and **finishes** (and logs errors if it fails).
- `@log_time` → measures and prints the **execution time**.

`functools.wraps` keeps the original function's name and docstring, so `func.__name__` still says `read_csv` instead of `wrapper`.

In [2]:
def log_action(func):
    """Log when a function/method starts and finishes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"▶ START  {func.__qualname__}")
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            logger.error(f"✖ FAILED {func.__qualname__}: {e}")
            raise
        logger.info(f"✔ FINISH {func.__qualname__}")
        return result
    return wrapper


def log_time(func):
    """Measure and print how long a function/method takes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱ {func.__qualname__} took {elapsed * 1000:.3f} ms")
        return result
    return wrapper

### Quick test on a plain function

In [3]:
@log_action
@log_time
def slow_add(a, b):
    time.sleep(0.2)
    return a + b

print("Result:", slow_add(2, 3))
print("Name preserved by functools.wraps:", slow_add.__name__)

09:23:11 | INFO | ▶ START  slow_add


⏱ slow_add took 200.203 ms
09:23:12 | INFO | ✔ FINISH slow_add


Result: 5
Name preserved by functools.wraps: slow_add


## Step 2 — Apply the decorators to `CSVReader` and `DataCleaner`

The class logic is exactly the same as before — the only change is the decorator lines above each method. That is the Open/Closed Principle in action.

In [4]:
class CSVReader:
    """Responsible only for loading CSV data."""

    def __init__(self, file_path):
        self.file_path = file_path

    @log_action
    @log_time
    def read_csv(self):
        df = pd.read_csv(self.file_path)
        print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
        return df


class DataCleaner:
    """Responsible only for cleaning data."""

    def __init__(self, df):
        self.df = df.copy()

    @log_action
    @log_time
    def show_missing(self):
        missing = self.df.isnull().sum()
        print("Missing values per column:")
        print(missing[missing > 0])
        return missing

    @log_action
    @log_time
    def fill_missing_with_mean(self):
        num_cols = self.df.select_dtypes(include="number").columns
        self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].mean().round(1))
        return self.df

    @log_action
    @log_time
    def remove_duplicates(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        print(f"Removed {before - len(self.df)} duplicate rows")
        return self.df

## Step 3 — Run the pipeline and watch the logs

In [5]:
reader = CSVReader("sample_data.csv")
raw_df = reader.read_csv()
raw_df

09:23:12 | INFO | ▶ START  CSVReader.read_csv


Loaded 7 rows × 7 columns
⏱ CSVReader.read_csv took 2.641 ms
09:23:12 | INFO | ✔ FINISH CSVReader.read_csv


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [6]:
cleaner = DataCleaner(raw_df)
cleaner.show_missing()
cleaner.fill_missing_with_mean()
clean_df = cleaner.remove_duplicates()
clean_df

09:23:12 | INFO | ▶ START  DataCleaner.show_missing


Missing values per column:
age          3
height_cm    1
weight_kg    1
score        2
dtype: int64
⏱ DataCleaner.show_missing took 1.023 ms
09:23:12 | INFO | ✔ FINISH DataCleaner.show_missing


09:23:12 | INFO | ▶ START  DataCleaner.fill_missing_with_mean


⏱ DataCleaner.fill_missing_with_mean took 2.922 ms
09:23:12 | INFO | ✔ FINISH DataCleaner.fill_missing_with_mean


09:23:12 | INFO | ▶ START  DataCleaner.remove_duplicates


Removed 0 duplicate rows
⏱ DataCleaner.remove_duplicates took 1.050 ms
09:23:12 | INFO | ✔ FINISH DataCleaner.remove_duplicates


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,30.2,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,86.0
3,4,David,30.2,167.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.2,162.0,60.0,Chicago,92.0


In [7]:
print("Missing values after cleaning:", clean_df.isnull().sum().sum())

Missing values after cleaning: 0


## Step 4 — Open/Closed Principle without touching the class

We can even add timing/logging to a class **we are not allowed to edit** by wrapping its method from the outside:

In [8]:
class Reporter:
    def summary(self, df):
        return df.describe().round(2)

# Extend behaviour from the outside — Reporter's source is unchanged
Reporter.summary = log_action(log_time(Reporter.summary))

Reporter().summary(clean_df)

09:23:12 | INFO | ▶ START  Reporter.summary


⏱ Reporter.summary took 7.039 ms
09:23:12 | INFO | ✔ FINISH Reporter.summary


,id,age,height_cm,weight_kg,score
count,7.00,7.00,7.00,7.00,7.00
mean,4.00,30.23,167.00,70.00,86.00
std,2.16,2.41,5.29,10.47,5.45
min,1.00,27.00,160.00,55.00,75.00
25%,2.50,29.50,163.50,64.00,85.50
50%,4.00,30.20,167.00,70.00,86.00
75%,5.50,30.20,170.00,76.00,89.00
max,7.00,35.00,175.00,85.00,92.00


## Summary

| Decorator | What it adds | Applied to |
|---|---|---|
| `@log_action` | Start / finish (and error) log messages | `read_csv`, `show_missing`, `fill_missing_with_mean`, `remove_duplicates` |
| `@log_time` | Execution time in ms | same methods |

**Why this follows Open/Closed:** logging and timing were added without modifying the bodies of `CSVReader` or `DataCleaner`. The decorators are reusable on any function, and removing them is as easy as deleting one line.

**Stacking order:** with `@log_action` on top of `@log_time`, `log_action` is the outer wrapper — its START prints first and FINISH prints last, and the timer measures only the method itself.